# 02 — Patch-graph construction diagnostics

This notebook is a thin Colab entry point. Reusable code lives in `src/cross_image_glot`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImageGLOT_repo")

if "<YOUR_GITHUB_USERNAME>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImageGLOT_repo
!pip install -q -r requirements.txt

%load_ext autoreload
%autoreload 2

In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

In [ ]:
from torch_geometric.data import Batch

from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import MiniImageNetFeatureDataset, FewShotFeatureEpisodeDataset
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder, validate_patch_graph

restore_feature_splits(["train"], paths.drive_feature_dir, paths.local_feature_dir)
train_features = MiniImageNetFeatureDataset(paths.local_feature_dir, "train", max_cached_shards=6)
train_episodes = FewShotFeatureEpisodeDataset(
    train_features, n_way=5, k_shot=5, queries_per_class=1,
    num_episodes=1000, seed=42, vary_by_epoch=True,
)
episode = train_episodes[0]
graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(train_features.metadata["grid_size"]), top_k=10,
    min_similarity=None, graph_dtype=torch.float32,
    similarity_device=device,
)

In [ ]:
graph = graph_builder.build_graph(
    query_patches=episode["query_patches"][0, 0],
    support_patches=episode["support_patches"][0],
    candidate_id=0,
    query_data_id=episode["query_indices"][0][0],
    support_data_ids=episode["support_indices"][0],
)
validate_patch_graph(graph, graph_builder)
print(graph)

In [ ]:
episode_graphs = graph_builder.build_episode_graphs(episode)
first_query_graphs = episode_graphs.graphs[:5]
batch = Batch.from_data_list(first_query_graphs)
print("Graphs:", batch.num_graphs)
print("x:", batch.x.shape)
print("edge_index:", batch.edge_index.shape)
print("edge_attr:", batch.edge_attr.shape)
print("candidate IDs:", batch.candidate_id)

In [ ]:
from cross_image_glot.models import PatchGraphSAGEEncoder

encoder = PatchGraphSAGEEncoder(input_dim=batch.x.shape[-1], hidden_dim=256, num_layers=2, dropout=0.1).to(device)
encoder.eval()
with torch.no_grad():
    single = encoder(first_query_graphs[0].clone().to(device))
    batched = encoder(batch.clone().to(device))
    inside_batch = batched[batch.batch == 0]
torch.testing.assert_close(single, inside_batch, rtol=1e-5, atol=1e-6)
print("Batch isolation passed.")